# 🧠 AI-Driven Mental Health Chatbot
**Built with:** LangChain · LLaMA 3.3 (Groq) · ChromaDB · HuggingFace Embeddings · Gradio

This chatbot uses Retrieval-Augmented Generation (RAG) to answer mental health questions from a provided PDF document.

---
**How it works:**
1. Loads a mental health PDF
2. Splits it into chunks and stores embeddings in ChromaDB
3. On user query, retrieves relevant chunks and sends to LLaMA 3.3 via Groq
4. Returns a compassionate, context-aware response via Gradio UI

In [36]:
# ── Step 1: Install Required Libraries ────────────────────────────────────────
# Run this cell first. It installs everything the chatbot needs.
!pip install -q langchain-groq langchain-community langchain-chroma langchain-text-splitters langchain-huggingface pypdf gradio

In [37]:
# ── Step 2: Import Libraries ───────────────────────────────────────────────────
# These are all the tools our chatbot needs
import os
from getpass import getpass

from langchain_groq import ChatGroq                                      # The LLM (AI brain)
from langchain_community.document_loaders import PyPDFLoader             # Reads PDF files
from langchain_chroma import Chroma                                      # Vector database
from langchain_huggingface import HuggingFaceEmbeddings                  # Converts text to numbers
from langchain_text_splitters import RecursiveCharacterTextSplitter      # Splits PDF into chunks
from langchain_core.prompts import PromptTemplate                        # Shapes how we ask the AI
from langchain_core.output_parsers import StrOutputParser                # Cleans up AI response
from langchain_core.runnables import RunnablePassthrough                 # Passes data through chain
import gradio as gr                                                       # Chat UI

print('✅ All imports successful!')

✅ All imports successful!


In [38]:
# ── Step 3: Configuration ──────────────────────────────────────────────────────
# Enter your Groq API key when prompted (it won't be visible as you type)
GROQ_API_KEY = getpass('🔑 Enter your Groq API Key: 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 ')

# ⚠️ UPDATE THIS: path to your mental health PDF in Google Drive
PDF_PATH = "/content/drive/MyDrive/mental_health_Document.pdf"

# Where to save the vector database (no need to change this)
DB_DIR = "./chroma_db"

🔑 Enter your Groq API Key: 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 ··········


In [39]:
# ── Step 4: Initialize the AI Model ───────────────────────────────────────────
# This connects to LLaMA 3.3 running on Groq's servers (fast + free)
llm = ChatGroq(
    temperature=0,              # 0 = consistent answers, 1 = more creative
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

print('✅ AI model ready!')

✅ AI model ready!


In [40]:
# ── Step 5: Build or Load the Vector Database ──────────────────────────────────
# A vector database stores your PDF as numbers so the AI can search it quickly

# This is the embedding model — it converts text into numbers
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def create_vector_db(pdf_path, db_dir):
    """Reads the PDF, splits it into chunks, and saves to ChromaDB."""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found at: {pdf_path}. Please update PDF_PATH in Step 3.")

    print("📄 Loading PDF...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"   Loaded {len(documents)} pages")

    print("✂️  Splitting into chunks...")
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(documents)
    print(f"   Created {len(chunks)} chunks")

    print("💾 Saving to vector database...")
    db = Chroma.from_documents(chunks, embedder, persist_directory=db_dir)
    print(f"✅ Vector database saved!")
    return db


# If database already exists, load it. Otherwise create a new one.
if os.path.isdir(DB_DIR) and os.listdir(DB_DIR):
    print("📂 Loading existing vector database...")
    vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embedder)
    print("✅ Vector database loaded!")
else:
    vector_db = create_vector_db(PDF_PATH, DB_DIR)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📂 Loading existing vector database...
✅ Vector database loaded!


In [41]:
# ── Step 6: Build the QA Chain ─────────────────────────────────────────────────
# This is the pipeline: User question → Find relevant chunks → Ask AI → Get answer

prompt_template = """You are a compassionate and knowledgeable mental health assistant.
Use the context below to answer the user's question in a warm and supportive way.
If the answer is not in the context, say: "I don't have enough information on that,
but I encourage you to speak with a mental health professional."

Context: {context}
User: {question}
Assistant:"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

retriever = vector_db.as_retriever(search_kwargs={"k": 5})  # Fetch top 5 relevant chunks

# Chain: retrieve context + pass question → fill prompt → send to LLM → parse output
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

print('✅ QA Chain ready!')

✅ QA Chain ready!


In [42]:
# ── Step 7: Chatbot Response Function ─────────────────────────────────────────
def chatbot_response(user_input, history):
    """Takes user's question and returns AI response."""
    if not user_input.strip():
        return "Please type a question to get started 😊"
    try:
        return qa_chain.invoke(user_input)
    except Exception as e:
        return f"Something went wrong: {str(e)}"

In [43]:
# ── Step 8: Launch Gradio Chat UI ─────────────────────────────────────────────
# This opens a chat interface — click the public URL to open it in your browser
app = gr.ChatInterface(
    fn=chatbot_response,
    type="messages",
    title="🧠 Mental Health Chatbot",
    description="Ask me anything about mental health. I'm here to help with compassion and care.",
    examples=[
        "What is anxiety and how can I manage it?",
        "How can I improve my sleep for better mental health?",
        "What are some stress relief techniques?"
    ],
    theme="soft"
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b79714d468e1cfa067.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
